In [ ]:
import sklearn.linear_model


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

data=pd.read_csv("housing.csv")
data.info()

In [ ]:
data.head()


In [ ]:
data.describe()
data

In [ ]:
data["median_income"]

In [ ]:
data.hist(bins=50, figsize=(20,15)) # plotting eah numerical attributes with its occurance in y=axis
plt.show()


In [ ]:
# creation of test set using a function

def split_train_test(data,test_ratio):
    shuffled_indices=np.random.permutation(len(data))
    test_size=int(len(data)*test_ratio)
    test_indices=shuffled_indices[:test_size]
    train_indices=shuffled_indices[test_size:]
    return data.iloc[train_indices], data.iloc[test_indices]


train_set ,test_set =split_train_test(data,0.2)

print(len(train_set), len(test_set))
print(train_set)

In [ ]:
from sklearn.model_selection import train_test_split
Train_set, test_set = train_test_split( data,test_size=0.2,random_state=42)
print(Train_set)
print(test_set)

In [ ]:
data["income_cat"]=pd.cut(data["median_income"],bins=[0.,1.5,3.0,4.5,6., np.inf], labels=[1,2,3,4,5])
data["income_cat"].hist()



In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

split= StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(data, data["income_cat"]):
    strat_train_set=data.loc[train_index]
    strat_test_set=data.loc[test_index]


for set_  in (strat_train_set,strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)




In [ ]:
# create a copy of train data while leaves aside a test data
# look the housing distribution

data_train= strat_train_set.copy()
data_train.plot(kind="scatter", x="longitude", y="latitude", alpha=0.1)


In [ ]:
# looking for housing distribution with thier prices

data_train.plot(kind="scatter", x="longitude",y="latitude", alpha=0.4, s=data_train["population"]/100, label="population", figsize=(10,7),c="median_house_value", cmap=plt.get_cmap("jet"), colorbar=True, )
plt.legend()

In [ ]:
# we correlate different attributes to see how their related 
corr_matrix=data_train.corr(numeric_only=True)
corr_matrix["median_house_value"].sort_values(ascending=False)

In [ ]:
# Another ways of checck correlation is to use pand's scatter matrix by consider few attributes


from pandas.plotting import scatter_matrix
attributes=["median_house_value", "median_income", "total_rooms" ,"housing_median_age"]

scatter_matrix(data_train[attributes], figsize=(12, 8))
plt.show()

In [ ]:
data_train.plot(kind="scatter",x="median_income", y="median_house_value", alpha=0.1)

In [ ]:

#combinations of Representatives logical Attributes:

data_train["rooms_per_household"] = data_train["total_rooms"]/data_train["households"]
data_train["bedrooms_per_room"] = data_train["total_bedrooms"]/data_train["total_rooms"]
data_train["population_per_household"]=data_train["population"]/data_train["households"]

corr_matrix=data_train.corr(numeric_only=True)
corr_matrix["median_house_value"].sort_values(ascending=False)

In [ ]:
# Prepare the data for machine learning Algorithms
data=strat_train_set.drop("median_house_value",axis=1)
data_labels=strat_train_set["median_house_value"].copy()
display(data_labels) 



In [ ]:
# handle missing value using a sklearn library Imputer
from sklearn.impute import SimpleImputer
imputer=SimpleImputer(strategy="median")

# creat the set with no non numerical attributes

data_num=data.drop("ocean_proximity", axis=1)
imputer.fit(data_num)
X=imputer.transform(data_num)
data_tr=pd.DataFrame(X, columns=data_num.columns)
data_tr.info() # All data have no missing values

In [ ]:
data_cat=data[["ocean_proximity"]]
data_cat.head(10)

In [ ]:
# Since the ML algorithms work better with numbers , Lets converts these attributes from text to numbers
from sklearn.preprocessing import OrdinalEncoder
Ordinal_encoder=OrdinalEncoder()
data_cat_encoded=Ordinal_encoder.fit_transform(data_cat)
data_cat_encoded[:50]


In [47]:
# since the ML algorithms assume number are in order , then it should match with text relations
from sklearn.preprocessing import OneHotEncoder
import numpy as np
cat_encoder=OneHotEncoder()
data_cat_1hot=cat_encoder.fit_transform(data_cat)
cat_encoder.categories_

[array(['<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'],
       dtype=object)]